### Explore A Jain's IEI pipeline
### Julian Moran
### 2026-02-09

In [34]:
import os

from dotenv import load_dotenv

import polars as pl
import numpy as np

# Env
pl.Config.set_tbl_rows(10)
load_dotenv("../.env")
INSTALL_PATH = os.environ["INSTALL_PATH"]

# Pipeline IO

1. DefenseFinder RefSeq IDs --> UniProt IDs
    - DefenseFinder: db of all bacterial genes implicated in antiphage systems
    - RefSeq bacterial accession IDs:   denote a bacterial immunoprotein; often begin with `WP`
    - UniProt access IDs:               denote a protein; e.g. `A0A7P0TAE1_HUMAN`

<br>

2. UniProt IDs --> Foldseek cluster ID

<br>

3. Foldseek cluster ID --> related human cluster ID

<br>

4. UniProt IDs --> UniprotKB annotations

<br>


# Questions

1. How does `01_RefSeq2Uniprot_mapping/` fit into the pipeline?
    - Are data in `01_RefSeq2Uniprot_mapping/` the output from pipeline step 2.?
    - Are data in `griid_gene_list/`, `gasdermin/`, `vipirin/`, `zorya/` prioritized subsets from `01_RefSeq2Uniprot_mapping/`?
    - If so, why are the subsets substantially larger than the original data?
    - `WP_063110969.1`, from `gasdermin/2025-07-26_gasdermin_annotated.xlsx`, is not in `data_local/01_RefSeq2Uniprot_mapping/2025-06-24_RefSeq2Uniprot_mapping.xlsx`

<br>

2. What is the relationship between each UniProt accession number and its corresponding cluster?
    - i.e. in datasets like `zorya/2025-07-27_zorya_annotated.tsv`?


<br>

3. Why do we see identical rows in the annotated data?
    - best to denote rows as `RefSeq_ID`,`UniProt_ID`,`FoldSeq_cluster`,`Similar_human_gene` 
    - e.g. in `zorya/2025-07-27_zorya_annotated.tsv`, see multiple rows for `WP_063192462.1`,`A0A142CZW3`,`A0A142CZW3`,`B4DLD8`


<br>

# Filtering criteria

1. Select for human proteins in same cluster as bacterial
    - `defense_system_cluster_id == cluster_id`
    - in annotated data, `defense_system_cluster_id` denotes the FoldSeek cluster that contains the query DefenseFinder protein
    - in annotated data, `cluster_id` denotes the most-similar target FoldSeek cluster containing a human protein 


2. Select for human proteins closest in AA length to each bacterial protein


2. Select for closest human genes that are in EAGLE definitive



<br>

# Missing read permissions

1. `00_run_pipeline.sh`
2. `scripts/05_generate_slurm_file.py`
3. `scripts/06_find_human_foldseek_matches.py`
4. `scripts/07_run_foldseek.py`
5. `scripts/07_submit_run_foldseek.sh`
6. `scripts/08_add_annotations-optimized-new_metrics.py`

In [4]:
# ============================================================
#                             Args
# ============================================================

args = {
    "data_file_foldseek_uniprot": f"{INSTALL_PATH}/data_local/02_FoldSeek_Results_w_Uniprot_matches/2025-07-26_batch_0_annotated.tsv",
    "data_file_GRIID": f"{INSTALL_PATH}/data_local/griid_gene_list/2025-07-30_data_filtered_for_griid_genes.tsv",
    "data_file_gasdermin": f"{INSTALL_PATH}/data_sync/gasdermin/2025-07-26_gasdermin_annotated.tsv",
    "data_file_vipirin": f"{INSTALL_PATH}/data_sync/viperin/2025-07-26_viperin_annotated.tsv",
    "data_file_zorya": f"{INSTALL_PATH}/data_local/zorya/2025-07-27_zorya_annotated.tsv"
}

In [20]:
# ============================================================
#                             In
# ============================================================

data_ann = {}
for key, path in args.items():
    if "data_file_" in key and not "foldseek" in key:
        data_key = key.replace("file_", "")
        data_ann[data_key] = pl.read_csv(
            args[key],
            has_header=True,
            separator="\t",
            schema_overrides={
                "duplicate_count": pl.Int64,
                "rank": pl.Int64,
                "evalue": pl.Float64,
                "cluFlag": pl.Int64,
                "fident": pl.Float64,
                "alnlen": pl.Int64,
                "mismatch": pl.Int64,
                "gapopen": pl.Float64,
                "qstart": pl.Int64,
                "qend": pl.Int64,
                "tstart": pl.Int64,
                "tend": pl.Int64,
                "evalue_foldseek": pl.Float64,
                "bits": pl.Int64,
                "Query_length": pl.Int64,
                "Human_prot_total_length": pl.Int64,
                "Human_domain_percentage": pl.Float64,
                "Query_percentage": pl.Float64,
                "Query_overlap_w_Human_protein": pl.Float64,
                "Target_length": pl.Int64,
                "Bacterial_prot_total_length": pl.Int64,
                "Bacterial_domain_percentage": pl.Float64,
                "Target_percentage": pl.Float64,
                "Target_overlap_w_Bacterial_protein": pl.Float64,
                "Bacterial_Human_Len_Ratio": pl.Float64,
                "Query_Target_Overlap_Length": pl.Float64,
                "Overlap_ratio": pl.Float64,
            },
            infer_schema_length=10000
        )

df_foldseek_uniprot = pl.read_csv(
    args["data_file_foldseek_uniprot"],
    separator="\t",
    has_header=True
)

df_foldseek_uniprot[0:19]

accession_in_sys,type,subtype,species,sys_id,sys_beg,sys_end,duplicate_count,rank,defense_system_protein_id,defense_system_cluster_id,cluster_id,protein_id,evalue,cluFlag,fident,alnlen,mismatch,gapopen,qstart,qend,tstart,tend,evalue_foldseek,bits,Reviewed,Entry Name,Protein names,Gene Names,Organism,Gene Names (ordered locus),Gene Names (ORF),Gene Names (primary),Gene Names (synonym),Proteomes,Gene Ontology (biological process),Gene Ontology (cellular component),Gene Ontology (GO),Gene Ontology (molecular function),Gene Ontology IDs,CDD,FunFam,Gene3D,InterPro,PANTHER,Pfam,PROSITE,SMART,RefSeq,Query_range,Query_length,Human_prot_total_length,Human_prot_domain_match,Human_domain_percentage,Query_percentage,Query_overlap_w_Human_protein,Human_domain_note,Human_domain_evidence,Target_range,Target_length,Bacterial_prot_total_length,Bacterial_prot_domain_match,Bacterial_domain_percentage,Target_percentage,Target_overlap_w_Bacterial_protein,Bacterial_domain_note,Bacterial_domain_evidence,Bacterial_Human_Len_Ratio,Query_Target_Overlap_Length,Overlap_ratio
str,str,str,str,str,str,str,i64,i64,str,str,str,str,f64,i64,f64,i64,i64,i64,i64,i64,i64,i64,f64,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,i64,str,f64,f64,f64,str,str,str,i64,i64,str,f64,f64,f64,str,str,f64,i64,f64
"""WP_001223208.1""","""MazEF""","""MazEF""","""Shigella dysenteriae,Escherich…","""GCF_003017995_NZ_CP027368_MazE…","""GCF_002156845.1_NZ_CP021339_05…","""GCF_019428585.1_NZ_CP080119_01…",671,6,"""A0A024L8H8""","""S3FMY0""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""nan..nan""",null,null,null,null,null,null,null,null,"""nan..nan""",null,83,null,null,null,null,null,null,null,null,null
"""WP_003414296.1""","""Cas""","""CAS_Class1-Subtype-III-A""","""Mycobacterium tuberculosis,Myc…","""GCF_014884645_NZ_CP043996_CAS_…","""GCF_002975475.1_NZ_CP027035_02…","""GCF_013010385.1_NZ_CP053092_02…",264,23,"""A0A045ICU9""","""A0A1I6JKI6""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""nan..nan""",null,null,null,null,null,null,null,null,"""nan..nan""",null,124,null,null,null,null,null,null,null,null,null
"""WP_002885155.1""","""AbiE""","""AbiE""","""Klebsiella sp. P1927,Klebsiell…","""GCF_022354545_NZ_CP055087_AbiE…","""GCF_022494255.1_NZ_CP058743_04…","""GCF_003286975.1_NZ_CP030172_04…",454,12,"""A0A060VEP4""","""J1I8N4""","""A0A6J2X4D6""","""Q5TF85""",0.02082,1,0.098,297,214,0,179,475,10,247,0.001215,42,"""unreviewed""","""Q5TF85_HUMAN""","""polynucleotide adenylyltransfe…","""TENT5A FAM46A hCG_401094""","""Homo sapiens (Human)""",null,"""hCG_401094""","""TENT5A""","""FAM46A""","""UP000005640: Chromosome 6""",null,null,"""poly(A) RNA polymerase activit…","""poly(A) RNA polymerase activit…","""GO:1990817""",null,null,null,"""IPR012937; TET5.;""","""PTHR12974; PRION-LIKE- Q/N-RIC…","""PF07984; NTP_transf_7; 1.;""",null,"""SM01153; DUF1693; 1.;""",null,"""179.0..475.0""",297,523,null,null,null,null,null,null,"""10.0..247.0""",238,305,null,null,null,null,null,null,0.583174,238,0.780328
"""WP_002885155.1""","""AbiE""","""AbiE""","""Klebsiella sp. P1927,Klebsiell…","""GCF_022354545_NZ_CP055087_AbiE…","""GCF_022494255.1_NZ_CP058743_04…","""GCF_003286975.1_NZ_CP030172_04…",454,12,"""A0A060VEP4""","""J1I8N4""","""G1SF99""","""Q5VWP2""",0.08138,1,0.094,295,265,0,49,343,10,302,0.0009699,39,"""reviewed""","""TET5C_HUMAN""","""Terminal nucleotidyltransferas…","""TENT5C FAM46C""","""Homo sapiens (Human)""",null,null,"""TENT5C""","""FAM46C""","""UP000005640: Chromosome 1""","""in utero embryonic development…","""centrosome [GO:0005813]; cytop…","""centrosome [GO:0005813]; cytop…","""poly(A) RNA polymerase activit…","""GO:0001701; GO:0003723; GO

In [35]:
data_ann["data_GRIID"]

accession_in_sys,type,subtype,species,sys_id,sys_beg,sys_end,duplicate_count,rank,defense_system_protein_id,defense_system_cluster_id,cluster_id,protein_id,evalue,cluFlag,fident,alnlen,mismatch,gapopen,qstart,qend,tstart,tend,evalue_foldseek,bits,Reviewed,Entry Name,Protein names,Gene Names,Organism,Gene Names (ordered locus),Gene Names (ORF),Gene Names (primary),Gene Names (synonym),Proteomes,Gene Ontology (biological process),Gene Ontology (cellular component),Gene Ontology (GO),Gene Ontology (molecular function),Gene Ontology IDs,CDD,FunFam,Gene3D,InterPro,PANTHER,Pfam,PROSITE,SMART,RefSeq,Query_range,Query_length,Human_prot_total_length,Human_prot_domain_match,Human_domain_percentage,Query_percentage,Query_overlap_w_Human_protein,Human_domain_note,Human_domain_evidence,Target_range,Target_length,Bacterial_prot_total_length,Bacterial_prot_domain_match,Bacterial_domain_percentage,Target_percentage,Target_overlap_w_Bacterial_protein,Bacterial_domain_note,Bacterial_domain_evidence,Bacterial_Human_Len_Ratio,Query_Target_Overlap_Length,Overlap_ratio,GRIID_gene
str,str,str,str,str,str,str,i64,i64,str,str,str,str,f64,i64,f64,i64,i64,f64,i64,i64,i64,i64,f64,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,i64,str,f64,f64,f64,str,str,str,i64,i64,str,f64,f64,f64,str,str,f64,f64,f64,str
"""WP_003734567.1""","""RM""","""RM_Type_II""","""Listeria monocytogenes""","""GCF_022869685_NZ_CP064373_RM_T…","""GCF_002025165.1_NZ_CP020022_02…","""GCF_002025165.1_NZ_CP020022_02…",14,189,"""A0A2Z5C2A5""","""A0A2E8RBP0""","""A0A2E8RBP0""","""A0A7P0TBC2""",0.0,1,0.157,393,308,0.0,72,464,2,367,1.7250e-15,345,"""unreviewed""","""A0A7P0TBC2_HUMAN""","""Heat shock protein 90 beta fam…","""HSP90B1""","""Homo sapiens (Human)""",null,null,"""HSP90B1""",null,"""UP000005640: Chromosome 12""",null,null,"""ATP binding [GO:0005524]; ATP …","""ATP binding [GO:0005524]; ATP …","""GO:0005524; GO:0016887; GO:005…","""cd16927; HATPase_Hsp90-like; 1…","""3.30.565.10:FF:000005; Heat sh…","""3.30.230.80; -; 1.;""3.30.565.1…","""IPR036890; HATPase_C_sf.;""IPR0…","""PTHR11528; HEAT SHOCK PROTEIN …","""PF13589; HATPase_c_3; 1.;""PF00…","""PS00298; HSP90; 1.;""","""SM00387; HATPase_c; 1.;""",null,"""72.0..464.0""",393,465,null,null,null,null,null,null,"""2.0..367.0""",366,641,null,null,null,null,null,null,1.378495,366.0,0.787097,"""Yes"""
"""WP_210422214.1""","""pAgo""","""pAgo_LongA""","""Dolichospermum sp. UHCC 0315A""","""GCF_008121535_NZ_CP043056_pAgo…","""GCF_008121535.1_NZ_CP043056_03…","""GCF_008121535.1_NZ_CP043056_03…",1,202,"""A0A5C0DWF7""","""A0A2A2KXD5""","""A0A2A2KXD5""","""Q9HCK5""",0.0,1,0.1,692,581,0.0,164,809,17,708,2.1320e-17,277,"""reviewed""","""AGO4_HUMAN""","""Protein argonaute-4 (Argonaute…","""AGO4 EIF2C4 KIAA1567""","""Homo sapiens (Human)""",null,null,"""AGO4""","""EIF2C4 KIAA1567""","""UP000005640: Chromosome 1""","""male gonad development [GO:000…","""cytoplasm [GO:0005737]; cytopl…","""cytoplasm [GO:0005737]; cytopl…","""double-stranded RNA binding [G…","""GO:0000932; GO:0003725; GO:000…","""cd02846; PAZ_argonaute_like; 1…","""2.170.260.10:FF:000001; Protei…","""3.40.50.2300; -; 1.;""2.170.260…","""IPR028604; AGO4.;""IPR014811; A…","""PTHR22891; EUKARYOTIC TRANSLAT…","""PF08699; ArgoL1; 1.;""PF16488; …","""PS50821; PAZ; 1.;""PS50822; PIW…","""SM01163; DUF1785; 1.;""SM00949;…","""NP_060099.2; NM_017629.3.;""","""164.0..809.0""",646,861,"""219..338""",100.0,18.58,75.03,"""['PAZ']""","""['ECO:0000255|PROSITE-ProRule:…","""17.0..708.0""",692,722,"""434..718""",96.49,39.74,95.84,"""['Piwi']""","""['ECO:0000259|SMART:SM00950']""",0.83856,646.0,0.894737,"""Yes"""
"""WP_210422214.1""","""pAgo""","""pAgo_LongA""","""Dolichospermum sp. UHCC 0315A""","""GCF_008121535_NZ_CP043056_pAgo…","""GCF_008121535.1_NZ_CP043056_03…","""GCF_008121535.1_NZ_CP043056_03…",1,202,"""A0A5C0DWF7""","""A0A2A2KXD5""","""A0A2A2KXD5""","""Q9HCK5""",0.0,1,0.1,692,581,0.0,164,809,17,708,2.1320e-17,277,"""reviewed""","""AGO4

In [37]:
data_ann["data_gasdermin"]

accession_in_sys,type,subtype,species,sys_id,sys_beg,sys_end,duplicate_count,rank,defense_system_protein_id,defense_system_cluster_id,cluster_id,protein_id,evalue,cluFlag,fident,alnlen,mismatch,gapopen,qstart,qend,tstart,tend,evalue_foldseek,bits,Reviewed,Entry Name,Protein names,Gene Names,Organism,Gene Names (ordered locus),Gene Names (ORF),Gene Names (primary),Gene Names (synonym),Proteomes,Gene Ontology (biological process),Gene Ontology (cellular component),Gene Ontology (GO),Gene Ontology (molecular function),Gene Ontology IDs,CDD,FunFam,Gene3D,InterPro,PANTHER,Pfam,PROSITE,SMART,RefSeq,Query_range,Query_length,Human_prot_total_length,Human_prot_domain_match,Human_domain_percentage,Query_percentage,Query_overlap_w_Human_protein,Human_domain_note,Human_domain_evidence,Target_range,Target_length,Bacterial_prot_total_length,Bacterial_prot_domain_match,Bacterial_domain_percentage,Target_percentage,Target_overlap_w_Bacterial_protein,Bacterial_domain_note,Bacterial_domain_evidence,Bacterial_Human_Len_Ratio,Query_Target_Overlap_Length,Overlap_ratio
str,str,str,str,str,str,str,i64,i64,str,str,str,str,f64,i64,f64,i64,i64,f64,i64,i64,i64,i64,f64,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,i64,str,f64,f64,f64,str,str,str,i64,i64,str,f64,f64,f64,str,str,f64,f64,f64
"""WP_063110969.1""","""GasderMIN""","""GasderMIN""","""Methylorubrum populi""","""GCF_002355515_NZ_AP014809_Gasd…","""GCF_002355515.1_NZ_AP014809_02…","""GCF_002355515.1_NZ_AP014809_02…",1,202,"""A0A160PE62""","""A0A2V2RJT9""","""A0A811ZZM7""","""A0A024RA58""",0.000024,1,0.116,277,240,0.0,1,277,9,280,0.000027,88,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""1.0..277.0""",277,null,null,null,null,null,null,null,"""9.0..280.0""",272,286,null,null,null,null,null,null,null,272.0,0.951049
"""WP_063110969.1""","""GasderMIN""","""GasderMIN""","""Methylorubrum populi""","""GCF_002355515_NZ_AP014809_Gasd…","""GCF_002355515.1_NZ_AP014809_02…","""GCF_002355515.1_NZ_AP014809_02…",1,202,"""A0A160PE62""","""A0A2V2RJT9""","""A0A811ZZM7""","""A0A2R8Y5J4""",0.000024,1,0.105,280,245,0.0,1,275,7,286,0.000062,92,"""unreviewed""","""A0A2R8Y5J4_HUMAN""","""Gasdermin C""","""GSDMC""","""Homo sapiens (Human)""",null,null,"""GSDMC""",null,"""UP000005640: Unplaced""","""programmed cell death [GO:0012…","""cytosol [GO:0005829]; plasma m…","""cytosol [GO:0005829]; plasma m…",null,"""GO:0005829; GO:0005886; GO:001…",null,null,null,"""IPR007677; Gasdermin.;""IPR0404…","""PTHR16399; GASDERMIN; 1.;""PTHR…","""PF04598; Gasdermin; 1.;""PF1770…",null,null,null,"""1.0..275.0""",275,508,null,null,null,null,null,null,"""7.0..286.0""",280,286,null,null,null,null,null,null,0.562992,275.0,0.961538
"""WP_063110969.1""","""GasderMIN""","""GasderMIN""","""Methylorubrum populi""","""GCF_002355515_NZ_AP014809_Gasd…","""GCF_002355515.1_NZ_AP014809_02…","""GCF_002355515.1_NZ_AP014809_02…",1,202,"""A0A160PE62""","""A0A2V2RJT9""","""A0A811ZZM7""","""A4FTY0""",0.000024,1,0.111,230,194,0.0,32,250,35,264,0.0001308,89,"""unreviewed""","""A4FTY0_HUMAN""","""Deafness, autosomal dominant 5""","""DFNA5""","""Homo sapiens (Human)""",null,null,"""DFNA5""",null,null,"""programmed cell death [GO:0012…","""cytoplasm [GO:0005737]; plasma…","""cytoplasm [GO:0005737]; plasma…",null,"""GO:0005737; GO:0005886; GO:001…",null,null,null,"""IPR040460; Gasdermin_pore.;""IP…","""PTHR15207:SF1; GASDERMIN-E; 1.…","""PF04598; Gasdermin; 1.;""PF1770…",null,null,null,"""32.0..250.0""",219,496,"""1..246""",87.4,98.17,44.15,"""['Gasdermin pore forming']""","""['ECO:0000259|Pfam:PF04598']""","""35.0..264.0""",230,286,null,null,null,null,null,null,0.576613,219.0,0.765734
"""WP_063110969.1""","""GasderMIN""","""GasderMIN""","""Methylorubrum populi""","""GCF_002355515_NZ_AP014809_Gasd…","""GCF_002355515.1_NZ_AP014809_02…","""GCF_002355515.1_NZ_AP014809_02…",1,202,"""A0A160PE62""","""A0A2V2RJT9""","""A0A811ZZM7""","""

In [40]:
data_ann["data_vipirin"]

accession_in_sys,type,subtype,species,sys_id,sys_beg,sys_end,duplicate_count,rank,defense_system_protein_id,defense_system_cluster_id,cluster_id,protein_id,evalue,cluFlag,fident,alnlen,mismatch,gapopen,qstart,qend,tstart,tend,evalue_foldseek,bits,Reviewed,Entry Name,Protein names,Gene Names,Organism,Gene Names (ordered locus),Gene Names (ORF),Gene Names (primary),Gene Names (synonym),Proteomes,Gene Ontology (biological process),Gene Ontology (cellular component),Gene Ontology (GO),Gene Ontology (molecular function),Gene Ontology IDs,CDD,FunFam,Gene3D,InterPro,PANTHER,Pfam,PROSITE,SMART,RefSeq,Query_range,Query_length,Human_prot_total_length,Human_prot_domain_match,Human_domain_percentage,Query_percentage,Query_overlap_w_Human_protein,Human_domain_note,Human_domain_evidence,Target_range,Target_length,Bacterial_prot_total_length,Bacterial_prot_domain_match,Bacterial_domain_percentage,Target_percentage,Target_overlap_w_Bacterial_protein,Bacterial_domain_note,Bacterial_domain_evidence,Bacterial_Human_Len_Ratio,Query_Target_Overlap_Length,Overlap_ratio
str,str,str,str,str,str,str,i64,i64,str,str,str,str,f64,i64,f64,i64,i64,f64,i64,i64,i64,i64,f64,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,i64,str,f64,f64,f64,str,str,str,i64,i64,str,f64,f64,f64,str,str,f64,f64,f64
"""WP_218920067.1""","""Viperin""","""Viperin""","""Chondromyces crocatus""","""GCF_001189295_NZ_CP012159_Vipe…","""GCF_001189295.1_NZ_CP012159_05…","""GCF_001189295.1_NZ_CP012159_05…",1,202,"""A0A0K1EKA3""","""A0A2E7NAI8""","""A0A0C2ICF2""","""B4DW16""",0.000003,2,0.128,234,185,0.0,20,253,44,256,1.1060e-8,189,"""unreviewed""","""B4DW16_HUMAN""","""cDNA FLJ57623, highly similar …",null,"""Homo sapiens (Human)""",null,null,null,null,null,"""tRNA processing [GO:0008033]""",null,"""4 iron, 4 sulfur cluster bindi…","""4 iron, 4 sulfur cluster bindi…","""GO:0008033; GO:0046872; GO:005…","""cd01335; Radical_SAM; 1.;""",null,"""3.20.20.70; Aldolase class I; …","""IPR013785; Aldolase_TIM.;""IPR0…","""PTHR13930; S-ADENOSYL-L-METHIO…","""PF04055; Radical_SAM; 1.;""PF08…","""PS51918; RADICAL_SAM; 1.;""",null,null,"""20.0..253.0""",234,346,"""14..258""",95.51,100.0,67.63,"""['Radical SAM core']""","""['ECO:0000259|PROSITE:PS51918'…","""44.0..256.0""",213,324,"""38..257""",96.82,100.0,65.74,"""['Radical SAM core']""","""['ECO:0000259|PROSITE:PS51918'…",0.936416,213.0,0.657407
"""WP_218920067.1""","""Viperin""","""Viperin""","""Chondromyces crocatus""","""GCF_001189295_NZ_CP012159_Vipe…","""GCF_001189295.1_NZ_CP012159_05…","""GCF_001189295.1_NZ_CP012159_05…",1,202,"""A0A0K1EKA3""","""A0A2E7NAI8""","""A0A0F9SLM6""","""J3QKL0""",0.05141,1,0.069,186,167,0.0,118,297,79,264,0.001903,58,"""unreviewed""","""J3QKL0_HUMAN""","""Hexosaminidase D (EC 3.2.1.52)…","""HEXD""","""Homo sapiens (Human)""",null,null,"""HEXD""",null,"""UP000005640: Chromosome 17""","""carbohydrate metabolic process…",null,"""beta-N-acetylhexosaminidase ac…","""beta-N-acetylhexosaminidase ac…","""GO:0004563; GO:0005975""","""cd06565; GH20_GcnA-like; 1.;""","""3.20.20.80:FF:000068; hexosami…","""3.20.20.80; Glycosidases; 1.;""","""IPR015883; Glyco_hydro_20_cat.…","""PTHR21040; BCDNA.GH04120; 1.;""…","""PF00728; Glyco_hydro_20; 1.;""",null,null,null,"""118.0..297.0""",180,544,"""63..205""",61.54,48.89,33.09,"""['Glycoside hydrolase family 2…","""['ECO:0000259|Pfam:PF00728']""","""79.0..264.0""",186,324,"""38..257""",81.36,96.24,57.41,"""['Radical SAM core']""","""['ECO:0000259|PROSITE:PS51918'…",0.595588,180.0,0.555556
"""WP_218920067.1""","""Viperin""","""Viperin""","""Chondromyces crocatus""","""GCF_001189295_NZ_CP012159_Vipe…","""GCF_001189295.1_NZ_CP012159_05…","""GCF_001189295.1_NZ_CP012159_05…",1,202,"""A0A0K1EKA3""","""A0A2E7NAI8""","""A0A0F9SLM6""","""Q8WVB3""",0.05141,1,0.081,139,109,0.0,118,256,79,198,0.01117,59,"""reviewed""","""HEXD_HUMAN""","""Hexosaminidase D (EC 3.2.1.52)…","""HEXD HEXDC""","""Homo sapiens (Human)""",null,null,"""HEXD""","""HEXDC

In [41]:
data_ann["data_zorya"]

accession_in_sys,type,subtype,species,sys_id,sys_beg,sys_end,duplicate_count,rank,defense_system_protein_id,defense_system_cluster_id,cluster_id,protein_id,evalue,cluFlag,fident,alnlen,mismatch,gapopen,qstart,qend,tstart,tend,evalue_foldseek,bits,Reviewed,Entry Name,Protein names,Gene Names,Organism,Gene Names (ordered locus),Gene Names (ORF),Gene Names (primary),Gene Names (synonym),Proteomes,Gene Ontology (biological process),Gene Ontology (cellular component),Gene Ontology (GO),Gene Ontology (molecular function),Gene Ontology IDs,CDD,FunFam,Gene3D,InterPro,PANTHER,Pfam,PROSITE,SMART,RefSeq,Query_range,Query_length,Human_prot_total_length,Human_prot_domain_match,Human_domain_percentage,Query_percentage,Query_overlap_w_Human_protein,Human_domain_note,Human_domain_evidence,Target_range,Target_length,Bacterial_prot_total_length,Bacterial_prot_domain_match,Bacterial_domain_percentage,Target_percentage,Target_overlap_w_Bacterial_protein,Bacterial_domain_note,Bacterial_domain_evidence,Bacterial_Human_Len_Ratio,Query_Target_Overlap_Length,Overlap_ratio
str,str,str,str,str,str,str,i64,i64,str,str,str,str,f64,i64,f64,i64,i64,f64,i64,i64,i64,i64,f64,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,i64,str,f64,f64,f64,str,str,str,i64,i64,str,f64,f64,f64,str,str,f64,f64,f64
"""WP_063192462.1""","""Zorya""","""Zorya_TypeI""","""Geobacillus sp. JS12""","""GCF_001592395_NZ_CP014749_Zory…","""GCF_001592395.1_NZ_CP014749_00…","""GCF_001592395.1_NZ_CP014749_00…",1,202,"""A0A142CZW3""","""A0A355UP19""","""A0A355UP19""","""B4DLD8""",0.0,1,0.21,662,519,0.0,277,934,202,863,3.9580e-33,807,"""unreviewed""","""B4DLD8_HUMAN""","""DNA helicase (EC 3.6.4.12)""",null,"""Homo sapiens (Human)""",null,null,null,null,null,"""chromatin organization [GO:000…","""nucleus [GO:0005634]""","""nucleus [GO:0005634]; ATP bind…","""ATP binding [GO:0005524]; DNA …","""GO:0003677; GO:0004386; GO:000…","""cd18668; CD1_tandem_CHD5-9_lik…","""2.40.50.40:FF:000001; chromodo…","""2.40.50.40; -; 2.;""3.40.50.300…","""IPR051493; CHD.;""IPR016197; Ch…","""PTHR46850; CHROMODOMAIN-HELICA…","""PF00385; Chromo; 2.;""PF00271; …","""PS50013; CHROMO_2; 1.;""PS00690…","""SM00298; CHROMO; 2.;""SM00487; …",null,"""277.0..934.0""",658,1014,"""353..396""",100.0,6.69,64.89,"""['Chromo']""","""['ECO:0000259|PROSITE:PS50013'…","""202.0..863.0""",662,874,"""698..851""",100.0,23.26,75.74,"""['Helicase C-terminal']""","""['ECO:0000259|PROSITE:PS51194'…",0.861933,658.0,0.75286
"""WP_063192462.1""","""Zorya""","""Zorya_TypeI""","""Geobacillus sp. JS12""","""GCF_001592395_NZ_CP014749_Zory…","""GCF_001592395.1_NZ_CP014749_00…","""GCF_001592395.1_NZ_CP014749_00…",1,202,"""A0A142CZW3""","""A0A355UP19""","""A0A355UP19""","""B4DLD8""",0.0,1,0.21,662,519,0.0,277,934,202,863,3.9580e-33,807,"""unreviewed""","""B4DLD8_HUMAN""","""DNA helicase (EC 3.6.4.12)""",null,"""Homo sapiens (Human)""",null,null,null,null,null,"""chromatin organization [GO:000…","""nucleus [GO:0005634]""","""nucleus [GO:0005634]; ATP bind…","""ATP binding [GO:0005524]; DNA …","""GO:0003677; GO:0004386; GO:000…","""cd18668; CD1_tandem_CHD5-9_lik…","""2.40.50.40:FF:000001; chromodo…","""2.40.50.40; -; 2.;""3.40.50.300…","""IPR051493; CHD.;""IPR016197; Ch…","""PTHR46850; CHROMODOMAIN-HELICA…","""PF00385; Chromo; 2.;""PF00271; …","""PS50013; CHROMO_2; 1.;""PS00690…","""SM00298; CHROMO; 2.;""SM00487; …",null,"""277.0..934.0""",658,1014,"""353..396""",100.0,6.69,64.89,"""['Chromo']""","""['ECO:0000259|PROSITE:PS50013'…","""202.0..863.0""",662,874,"""698..851""",100.0,23.26,75.74,"""['Helicase C-terminal']""","""['ECO:0000259|PROSITE:PS51194'…",0.861933,658.0,0.75286
"""WP_063192462.1""","""Zorya""","""Zorya_TypeI""","""Geobacillus sp. JS12""","""GCF_001592395_NZ_CP014749_Zory…","""GCF_001592395.1_NZ_CP014749_00…","""GCF_001592395.1_NZ_CP014749_00…",1,202,"""A0A142CZW3""","""A0A355UP19""","""A0A355UP19""","""B4DLD8""",0.0,1,0.21,662,519,0.0,277,934,202,863,3.9580e-33,807,"""unreviewe